# Brain tumor classifier — Colab quickstart

### Three steps

1. **Runtime → Change runtime type → GPU** (recommended).
2. **Run all** (or run cells in order once per session).
3. In the **Configuration** cell, set `PROCESSED_DATA_PATH` to the folder on Drive that contains **`Training/`** and **`Testing/`** (same layout as local `data/processed`). Leave it `""` if you will upload or copy data into the clone path later.

### Open in Colab

Use this link (branch **`google-colab`** has the latest Colab helpers):

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadSaljooq/Tumor-AI-training-model/blob/google-colab/notebooks/Colab_Quickstart.ipynb)


## Configuration (edit here)

Only the first group usually needs changes.

In [ ]:
# ------------- EDIT THIS BLOCK ONLY -------------
REPO_URL = "https://github.com/MuhammadSaljooq/Tumor-AI-training-model.git"
REPO_BRANCH = "google-colab"
CLONE_DIR = "/content/brain_tumor_classifier"

# Path to processed MRI data: must contain Training/<class>/ and Testing/<class>/
# Example: "/content/drive/MyDrive/brain_mri/processed"
PROCESSED_DATA_PATH = ""  # leave "" if no Drive data yet

# Optional: raw Kaggle-style class folders; pipeline can preprocess into data/processed
RAW_DATA_PATH = ""

MOUNT_GOOGLE_DRIVE = True
# -----------------------------------------------

print("Clone:", CLONE_DIR, "| branch:", REPO_BRANCH)
print("Processed path:", PROCESSED_DATA_PATH or "(not set)")


## Mount Google Drive (one-time auth per session)

Skip if `MOUNT_GOOGLE_DRIVE` is False and your data lives under `/content/...`.

In [ ]:
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Drive mount skipped — use paths under /content or turn MOUNT_GOOGLE_DRIVE on.")


## Bootstrap: clone, install, symlink data, Colab config

Re-running this cell is safe (existing clone is updated; symlinks can be replaced with `--force-link`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path(CLONE_DIR).resolve()


def run_cmd(args: list) -> None:
    readable = " ".join(str(a) for a in args)
    print("+", readable, flush=True)
    subprocess.check_call(args)


if not (ROOT / "main.py").is_file():
    if ROOT.exists() and not (ROOT / ".git").is_dir():
        raise RuntimeError(
            f"{ROOT} exists but is not this repo — remove/rename it or change CLONE_DIR"
        )
    run_cmd(["git", "clone", "-b", REPO_BRANCH, "--depth", "1", REPO_URL, str(ROOT)])
elif (ROOT / ".git").is_dir():
    print("Updating existing clone...", flush=True)
    run_cmd(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REPO_BRANCH])
    run_cmd(["git", "-C", str(ROOT), "checkout", REPO_BRANCH])
    run_cmd(["git", "-C", str(ROOT), "pull", "origin", REPO_BRANCH])

os.chdir(ROOT)
os.environ["BRAIN_TUMOR_CLASSIFIER_ROOT"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])

setup_args = [
    sys.executable,
    str(ROOT / "scripts" / "setup_colab.py"),
    "--skip-clone",
    "--skip-install",
    "--project-root",
    str(ROOT),
    "--write-colab-config",
    "--quiet-pip",
]
if PROCESSED_DATA_PATH:
    setup_args += ["--processed-data", str(PROCESSED_DATA_PATH), "--force-link"]
if RAW_DATA_PATH:
    setup_args += ["--raw-data", str(RAW_DATA_PATH), "--force-link"]

run_cmd(setup_args)

try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("cd", str(ROOT))
except Exception:
    pass

print("\nReady — project root:", ROOT)
print("Next: run the training cell below (or main.py with your own flags).")


## Train (starter command)

Edit `MODELS` or change `configs/config_colab.yaml` / `configs/config.yaml` for epochs and batch size.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path(os.environ.get("BRAIN_TUMOR_CLASSIFIER_ROOT", "/content/brain_tumor_classifier")).resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

MODELS = ["resnet50"]  # e.g. ["resnet50", "vit", "hybrid"]

cfg_path = ROOT / "configs" / "config_colab.yaml"
if not cfg_path.is_file():
    cfg_path = ROOT / "configs" / "config.yaml"

has_processed = (ROOT / "data" / "processed").exists()
args = [
    sys.executable,
    "main.py",
    "--config",
    str(cfg_path.relative_to(ROOT)),
    "--models",
    *MODELS,
]
if has_processed:
    args.append("--skip_preprocessing")
else:
    print("No data/processed — running preprocessing from raw (needs data/raw).")

print("+", " ".join(args))
subprocess.check_call(args, cwd=str(ROOT))


## Optional: experiment notebook

Open `notebooks/experiments.ipynb` from the file tree or run `from src...` imports — `BRAIN_TUMOR_CLASSIFIER_ROOT` is already set and cwd is the repo.